In [19]:
using LowLevelFEM
using StaticArrays
using Tensors

In [20]:
#=
#export energyMaterial, tensorToMatrix, tensorToVector

"""
    energyMaterial(ψ, C::TensorField; params=NamedTuple())

Compute the second Piola-Kirchhoff stress and material tangent from a strain
energy density function.

The function returns a named tuple:

    (; S, D)

where `S` is a `TensorField` and `D` is a 6×6 matrix whose entries are
`ScalarField`s. The matrix `D` can be used directly as a DSL coefficient matrix.
"""
function energyMaterial(ψ, C::TensorField; params=NamedTuple())
    problem = C.model
    Cn = elementsToNodes(C)

    nsteps = Cn.nsteps
    non = problem.non

    Sdata = zeros(9non, nsteps)
    Ddata = [zeros(non, nsteps) for _ in 1:6, _ in 1:6]

    pnodes = _prepare_energy_params(params, problem)

    @inbounds for it in 1:nsteps
        for a in 1:non
            base = 9(a - 1)

            Cmat = @SMatrix [
                Cn.a[base+1, it] Cn.a[base+4, it] Cn.a[base+7, it]
                Cn.a[base+2, it] Cn.a[base+5, it] Cn.a[base+8, it]
                Cn.a[base+3, it] Cn.a[base+6, it] Cn.a[base+9, it]
            ]

            p = _params_at_node(pnodes, a, it)

            Smat = LowLevelFEM.stress_from_energy(ψ, Cmat, p)
            Dmat = LowLevelFEM.tangent_from_energy(ψ, Cmat, p)

            Sdata[base+1, it] = Smat[1, 1]
            Sdata[base+2, it] = Smat[2, 1]
            Sdata[base+3, it] = Smat[3, 1]
            Sdata[base+4, it] = Smat[1, 2]
            Sdata[base+5, it] = Smat[2, 2]
            Sdata[base+6, it] = Smat[3, 2]
            Sdata[base+7, it] = Smat[1, 3]
            Sdata[base+8, it] = Smat[2, 3]
            Sdata[base+9, it] = Smat[3, 3]

            for i in 1:6, j in 1:6
                Ddata[i, j][a, it] = Dmat[i, j]
            end
        end
    end

    S = TensorField([], Sdata, Cn.t, [], nsteps, :tensor, problem)

    D = Matrix{ScalarField}(undef, 6, 6)
    for i in 1:6, j in 1:6
        D[i, j] = ScalarField([], Ddata[i, j], Cn.t, [], nsteps, :scalar, problem)
    end

    return (; S, D)
end

@inline function _prepare_energy_params(params::NamedTuple, problem::Problem)
    names = keys(params)
    vals = values(params)

    newvals = map(vals) do v
        if v isa Number
            v
        elseif v isa ScalarField
            elementsToNodes(v)
        else
            error("energyMaterial: parameters must be Number or ScalarField, got $(typeof(v)).")
        end
    end

    return NamedTuple{names}(newvals)
end

@inline function _param_value_at_node(v::Number, a::Int, it::Int)
    return v
end

@inline function _param_value_at_node(v::ScalarField, a::Int, it::Int)
    return v.a[a, it]
end

@inline function _params_at_node(params::NamedTuple, a::Int, it::Int)
    names = keys(params)
    vals = values(params)

    newvals = map(vals) do v
        _param_value_at_node(v, a, it)
    end

    return NamedTuple{names}(newvals)
end
=#

#=
"""
    tensorToMatrix(T::TensorField)

Convert a TensorField to a matrix of ScalarField components.
"""
function tensorToMatrix(T::TensorField)
    return [
        T[1, 1] T[1, 2] T[1, 3]
        T[2, 1] T[2, 2] T[2, 3]
        T[3, 1] T[3, 2] T[3, 3]
    ]
end
=#

#=
"""
    tensorToVector(T::TensorField)

Convert a symmetric TensorField to Voigt vector form:

    [xx, yy, zz, xy, yz, zx]
"""
function tensorToVector(T::TensorField)
    return [
        T[1, 1],
        T[2, 2],
        T[3, 3],
        T[1, 2],
        T[2, 3],
        T[3, 1],
    ]
end
=#

In [ ]:
#export energyMaterial, tensorToMatrix, tensorToVector, IIPioleKirchhoff, TagentMatrix

const VOIGT3D = ((1, 1), (2, 2), (3, 3), (1, 2), (2, 3), (3, 1))

#=
"""
    energyMaterial(ψ, C::TensorField; params=NamedTuple())

Compute the constitutive response from a strain energy density function.

The function evaluates the second Piola–Kirchhoff stress tensor and the
consistent material tangent matrix in a fully elementwise manner.

# Arguments
- `ψ`: strain energy density function of the form `ψ(C,p)`
- `C`: right Cauchy–Green tensor field (`TensorField`)
- `params`: named tuple containing material parameters

Material parameters may be:
- `Number`
- `ScalarField`

`ScalarField` parameters are internally converted to elementwise form if needed.

# Returns
A named tuple:

    (; S, D)

where:
- `S` is an elementwise `TensorField`
- `D` is a `6×6` matrix whose entries are elementwise `ScalarField`s

The tangent matrix `D` can be used directly as a DSL coefficient matrix.

# Notes
Voigt ordering:

    [xx, yy, zz, xy, yz, zx]

This function computes stress and tangent simultaneously for efficiency.
"""
function energyMaterial(ψ, C::TensorField; params=NamedTuple(), stress=true, tangent=true)
    problem = C.model
    Ce = isElementwise(C) ? C : nodesToElements(C)

    nsteps = Ce.nsteps
    nelem = length(Ce.numElem)

    pe = _prepare_energy_params_elementwise(params, Ce)

    S_A = Vector{Matrix{Float64}}(undef, nelem)

    D_A = [Vector{Matrix{Float64}}(undef, nelem) for _ in 1:6, _ in 1:6]

    @inbounds for eidx in 1:nelem
        elem = Ce.numElem[eidx]
        CAe = Ce.A[eidx]

        nnloc = size(CAe, 1) ÷ 9

        SAe = zeros(9nnloc, nsteps)

        DAe = [zeros(nnloc, nsteps) for _ in 1:6, _ in 1:6]

        for it in 1:nsteps
            for a in 1:nnloc
                base = 9(a - 1)

                Cmat = @SMatrix [
                    CAe[base+1, it] CAe[base+4, it] CAe[base+7, it]
                    CAe[base+2, it] CAe[base+5, it] CAe[base+8, it]
                    CAe[base+3, it] CAe[base+6, it] CAe[base+9, it]
                ]

                p = _params_at_element_node(pe, elem, a, it)

                Smat = LowLevelFEM.stress_from_energy(ψ, Cmat, p)
                Dmat = LowLevelFEM.tangent_from_energy(ψ, Cmat, p)

                SAe[base+1, it] = Smat[1, 1]
                SAe[base+2, it] = Smat[2, 1]
                SAe[base+3, it] = Smat[3, 1]
                SAe[base+4, it] = Smat[1, 2]
                SAe[base+5, it] = Smat[2, 2]
                SAe[base+6, it] = Smat[3, 2]
                SAe[base+7, it] = Smat[1, 3]
                SAe[base+8, it] = Smat[2, 3]
                SAe[base+9, it] = Smat[3, 3]

                for i in 1:6, j in 1:6
                    DAe[i, j][a, it] = Dmat[i, j]
                end
            end
        end

        S_A[eidx] = SAe

        for i in 1:6, j in 1:6
            D_A[i, j][eidx] = DAe[i, j]
        end
    end

    S = TensorField(S_A, [;;], Ce.t, Ce.numElem, nsteps, :tensor, problem)

    D = Matrix{ScalarField}(undef, 6, 6)
    for i in 1:6, j in 1:6
        D[i, j] = ScalarField(D_A[i, j], [;;], Ce.t, Ce.numElem, nsteps, :scalar, problem)
    end

    return (; S, D)
end
=#

@inline function _prepare_energy_params_elementwise(params::NamedTuple, C::TensorField)
    names = keys(params)
    vals = values(params)

    newvals = map(vals) do v
        if v isa Number
            v
        elseif v isa ScalarField
            ve = isElementwise(v) ? v : nodesToElements(v)

            if ve.nsteps != 1 && ve.nsteps != C.nsteps
                error("energyMaterial: parameter nsteps must be 1 or equal to C.nsteps.")
            end

            return Dict(zip(ve.numElem, ve.A))
        else
            error("energyMaterial: parameters must be Number or ScalarField, got $(typeof(v)).")
        end
    end

    return NamedTuple{names}(newvals)
end


@inline _param_value_at_element_node(v::Number, elem::Int, a::Int, it::Int) = v

@inline function _param_value_at_element_node(v::Dict, elem::Int, a::Int, it::Int)
    A = v[elem]
    itt = size(A, 2) == 1 ? 1 : it
    return A[a, itt]
end


@inline function _params_at_element_node(params::NamedTuple, elem::Int, a::Int, it::Int)
    names = keys(params)
    vals = values(params)

    newvals = map(vals) do v
        _param_value_at_element_node(v, elem, a, it)
    end

    return NamedTuple{names}(newvals)
end

"""
    tensorToVector(T::TensorField)

Convert a symmetric tensor field to Voigt vector form.

# Arguments
- `T`: symmetric tensor field

# Returns
A vector of `ScalarField`s in the ordering:

    [xx, yy, zz, xy, yz, zx]

# Examples
```julia
v = tensorToVector(S)
```
"""
function tensorToVector(T::TensorField)
    return [T[i, j] for (i, j) in VOIGT3D]
end

"""
    tensorToMatrix(T::TensorField)

Convert a tensor field to a matrix of scalar fields.

# Arguments
- `T`: tensor field

# Returns
A `3×3` matrix whose entries are `ScalarField`s corresponding to the tensor
components.

# Examples
```julia
M = tensorToMatrix(S)
```
The returned matrix can be used directly in DSL matrix-chain expressions.
"""
function tensorToMatrix(T::TensorField)
    return [
        T[1, 1] T[1, 2] T[1, 3]
        T[2, 1] T[2, 2] T[2, 3]
        T[3, 1] T[3, 2] T[3, 3]
    ]
end

tensorToMatrix

In [22]:
function _energyMaterial(
    ψ,
    C::TensorField;
    params=NamedTuple(),
    stress::Bool=true,
    tangent::Bool=true)
    !stress && !tangent && error("_energyMaterial: stress or tangent must be true.")

    problem = C.model
    Ce = isElementwise(C) ? C : nodesToElements(C)

    nsteps = Ce.nsteps
    nelem = length(Ce.numElem)

    pe = _prepare_energy_params_elementwise(params, Ce)

    S_A = stress ? Vector{Matrix{Float64}}(undef, nelem) : nothing
    D_A = tangent ? [Vector{Matrix{Float64}}(undef, nelem) for _ in 1:6, _ in 1:6] : nothing

    @inbounds for eidx in 1:nelem
        elem = Ce.numElem[eidx]
        CAe = Ce.A[eidx]

        nnloc = size(CAe, 1) ÷ 9

        SAe = stress ? zeros(9nnloc, nsteps) : nothing
        DAe = tangent ? [zeros(nnloc, nsteps) for _ in 1:6, _ in 1:6] : nothing

        for it in 1:nsteps
            for a in 1:nnloc
                base = 9(a - 1)

                Cmat = @SMatrix [
                    CAe[base+1, it] CAe[base+4, it] CAe[base+7, it]
                    CAe[base+2, it] CAe[base+5, it] CAe[base+8, it]
                    CAe[base+3, it] CAe[base+6, it] CAe[base+9, it]
                ]

                p = _params_at_element_node(pe, elem, a, it)

                if stress
                    Smat = LowLevelFEM.stress_from_energy(ψ, Cmat, p)

                    SAe[base+1, it] = Smat[1, 1]
                    SAe[base+2, it] = Smat[2, 1]
                    SAe[base+3, it] = Smat[3, 1]
                    SAe[base+4, it] = Smat[1, 2]
                    SAe[base+5, it] = Smat[2, 2]
                    SAe[base+6, it] = Smat[3, 2]
                    SAe[base+7, it] = Smat[1, 3]
                    SAe[base+8, it] = Smat[2, 3]
                    SAe[base+9, it] = Smat[3, 3]
                end

                if tangent
                    Dmat = LowLevelFEM.tangent_from_energy(ψ, Cmat, p)

                    for i in 1:6, j in 1:6
                        DAe[i, j][a, it] = Dmat[i, j]
                    end
                end
            end
        end

        if stress
            S_A[eidx] = SAe
        end

        if tangent
            for i in 1:6, j in 1:6
                D_A[i, j][eidx] = DAe[i, j]
            end
        end
    end

    S = stress ? TensorField(S_A, [;;], Ce.t, Ce.numElem, nsteps, :tensor, problem) : nothing

    D = if tangent
        DD = Matrix{ScalarField}(undef, 6, 6)
        for i in 1:6, j in 1:6
            DD[i, j] = ScalarField(D_A[i, j], [;;], Ce.t, Ce.numElem, nsteps, :scalar, problem)
        end
        DD
    else
        nothing
    end

    return (; S, D)
end

_energyMaterial (generic function with 1 method)

In [23]:
"""
    energyMaterial(ψ, C::TensorField; params=NamedTuple())

Compute the constitutive response from a strain energy density function.

The function evaluates the second Piola–Kirchhoff stress tensor and the
consistent material tangent matrix in a fully elementwise manner.

# Arguments
- `ψ`: strain energy density function of the form `ψ(C,p)`
- `C`: right Cauchy–Green tensor field (`TensorField`)
- `params`: named tuple containing material parameters

Material parameters may be:
- `Number`
- `ScalarField`

`ScalarField` parameters are internally converted to elementwise form if needed.

# Returns
A named tuple:

    (; S, D)

where:
- `S` is an elementwise `TensorField`
- `D` is a `6×6` matrix whose entries are elementwise `ScalarField`s

The tangent matrix `D` can be used directly as a DSL coefficient matrix.

# Notes
Voigt ordering:

    [xx, yy, zz, xy, yz, zx]

This function computes stress and tangent simultaneously for efficiency.
"""
function energyMaterial(ψ, C::TensorField; params=NamedTuple())
    return _energyMaterial(ψ, C; params=params, stress=true, tangent=true)
end

"""
    IIPiolaKirchhoff(ψ, C::TensorField; params=NamedTuple())

Compute the second Piola–Kirchhoff stress tensor from a strain energy density
function.

# Arguments
- `ψ`: strain energy density function of the form `ψ(C,p)`
- `C`: right Cauchy–Green tensor field (`TensorField`)
- `params`: named tuple containing material parameters

# Returns
An elementwise `TensorField` containing the second Piola–Kirchhoff stress.

# Notes
This function internally uses the same constitutive kernel as
`energyMaterial`, but computes only the stress tensor.
"""
function IIPiolaKirchhoff(ψ, C::TensorField; params=NamedTuple())
    return _energyMaterial(ψ, C; params=params, stress=true, tangent=false).S
end

"""
    TangentMatrix(ψ, C::TensorField; params=NamedTuple())

Compute the consistent material tangent matrix from a strain energy density
function.

# Arguments
- `ψ`: strain energy density function of the form `ψ(C,p)`
- `C`: right Cauchy–Green tensor field (`TensorField`)
- `params`: named tuple containing material parameters

# Returns
A `6×6` matrix whose entries are elementwise `ScalarField`s.

The returned matrix can be used directly as a DSL coefficient matrix.

# Notes
Voigt ordering:

    [xx, yy, zz, xy, yz, zx]

This function internally uses the same constitutive kernel as
`energyMaterial`, but computes only the tangent matrix.
"""
function TangentMatrix(ψ, C::TensorField; params=NamedTuple())
    return _energyMaterial(ψ, C; params=params, stress=false, tangent=true).D
end

TangentMatrix

In [24]:
structured_rect_mesh()

In [25]:
mat = Material("body", E=10, ν=0.4999)
prob = Problem([mat], type=:PlaneStress)
println("μ = ", mat.μ)
println("λ = ", mat.λ)
println("κ = ", mat.κ)

p = (μ=mat.μ, λ=mat.λ)

μ = 3.333555570371358
λ = 16664.444296288257
κ = 16666.6666666685


(μ = 3.333555570371358, λ = 16664.444296288257)

In [26]:
F = TensorField(prob, "body", [2 0 0; 0 1/2 0; 0 0 1])
C = F' * F

elementwise TensorField
[[4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;]  …  [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;], [4.0; 0.0; … ; 0.0; 1.0;;]]

In [27]:
ψ(C, p) = p.μ / 2 * (tr(C) - 3) - p.μ * log(sqrt(det(C))) +
          p.λ / 2 * log(sqrt(det(C)))^2

ψ (generic function with 1 method)

In [28]:
mat = energyMaterial(ψ, C; params=p)

S = mat.S      # TensorField
D = mat.D      # 6×6 Matrix, elemei ScalarField-ek

6×6 Matrix{ScalarField}:
 ScalarField([[1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;]  …  [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;]], Matrix{Float64}(undef, 0, 0), [0.0], [45, 46, 47, 48, 49, 50, 51, 52, 53, 54  …  135, 136, 137, 138, 139, 140, 141, 142, 143, 144], 1, :scalar, Problem("stru

In [29]:
S = IIPiolaKirchhoff(ψ, C, params=p)

elementwise TensorField
[[2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;]  …  [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;], [2.5001666777785188; 0.0; … ; 0.0; 0.0;;]]

In [30]:
D = TangentMatrix(ψ, C, params=p)

6×6 Matrix{ScalarField}:
 ScalarField([[1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;]  …  [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;], [1041.94; 1041.94; 1041.94; 1041.94;;]], Matrix{Float64}(undef, 0, 0), [0.0], [45, 46, 47, 48, 49, 50, 51, 52, 53, 54  …  135, 136, 137, 138, 139, 140, 141, 142, 143, 144], 1, :scalar, Problem("stru

In [31]:
tensorToMatrix(S)

3×3 Matrix{ScalarField}:
 ScalarField([[2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;]  …  [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;], [2.50017; 2.50017; 2.50017; 2.50017;;]], Matrix{Float64}(undef, 0, 0), [0.0], [45, 46, 47, 48, 49, 50, 51, 52, 53, 54  …  135, 136, 137, 138, 139, 140, 141, 142, 143, 144], 1, :scalar, Problem("stru

In [32]:
tensorToVector(S)

6-element Vector{ScalarField}:
 ScalarField([[2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;], [2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;], [2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;], [2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;], [2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;], [2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;], [2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;], [2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;], [2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;], [2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;]  …  [2.5001666777785188; 2.5001666777785188; 2.5001666777785188; 2.5001666777785188;;], [2.5001666777785188; 2.50016

In [33]:
probe(S, 0, 0, 0)

3×3 Matrix{Float64}:
 2.50017    0.0     0.0
 0.0      -10.0007  0.0
 0.0        0.0     0.0